# Abacus pz3 HEALPix and Density Diagnostics

This notebook summarizes the local diagnostics added while testing faster HEALPix pixel-neighbor construction and the pz3 galaxy-density deficit.

Issues caught:

- `jax-healpy` cannot currently be benchmarked in the active `ili-sbi` environment: the local checkout requires `jax >= 0.10.0`, while this environment has JAX 0.5.0. The `godmax-jaxhealpy` environment fixes the import issue and was used for CPU/GPU checks.
- In `godmax-jaxhealpy`, jax-healpy is numerically correct for the tested functions, but `query_disc` is not competitive for this tiny-disc pz3 workload. Batched/JIT `query_disc.vmap` took 213.8 s on CPU and 156.3 s on A100 for 200 queries, versus about 0.002 s for healpy.
- `healpy.query_disc(..., buff=...)` is correct for the tested pz3 samples, but it is not faster for the current workload. With the single-pixel shortcut enabled, plain `healpy` remains faster in the 50k random-halo benchmark.
- The pz3 galaxy-density deficit is not caused by stochastic HOD sampling or cap-mask bookkeeping. The sampled catalog matches the HOD mean; the HOD mean over halo centers in the true cap is already about 83% of the retained Stage-31 target.
- The exact angular cap and nside pixel-mask cap differ by 68 galaxies in the current nside1024 split product, so both are now reported explicitly.


In [ ]:
from pathlib import Path
import glob
import json
import pandas as pd

repo = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX')
meas = repo / 'data/xDESI/processed/abacus_backlight/stage31_pz3_cap600/measurements'

def load_json(name):
    with open(meas / name, 'r', encoding='utf-8') as f:
        return json.load(f)


## Pixel Builder Benchmarks

The current pz3 workload is dominated by halos that hit the single-pixel shortcut. The reusable `buff` path avoids repeated allocation inside `healpy.query_disc`, but our wrapper must copy the returned view before the next halo reuses the buffer. In these samples that copy/grow overhead is larger than the allocation savings.

In [ ]:
pixel_files = [
    'codex_pixel_work_healpy_50k.json',
    'codex_pixel_work_healpy_buff_50k.json',
    'codex_pixel_work_healpy_50k_noshortcut.json',
    'codex_pixel_work_healpy_buff_50k_noshortcut.json',
] + [Path(p).name for p in sorted(glob.glob(str(meas / 'pixel_work_scaling_*_job6484832.json')))]
rows = []
for name in pixel_files:
    payload = load_json(name)
    for row in payload['rows']:
        rows.append({
            'file': name,
            'backend': row['pixel_backend'],
            'workers': row['workers'],
            'shortcut_factor': row['single_pixel_angle_factor'],
            'query_disc_calls': row['n_query_disc'],
            'buffer_grows': row.get('n_query_disc_buffer_grows', 0),
            'runtime_s': row['runtime_s'],
            'halos_per_s': row['halos_per_s'],
            'pairs': row['n_pairs'],
        })
pd.DataFrame(rows).sort_values(['shortcut_factor', 'workers', 'backend'])


## jax-healpy Status

The diagnostic appends `/mnt/ceph/users/spandey/ltu-godmax/jax-healpy` to `sys.path`, sets CPU or CUDA JAX platform as requested, and records an explicit import-failure row if the package cannot load. The original `ili-sbi` run records the JAX API/version mismatch. The `godmax-jaxhealpy` runs show successful imports and zero mismatches for the tested pixel functions, but slow `query_disc` timings.

In [ ]:
jax_files = [
    'codex_healpix_function_jax_import_smoke.json',
    'codex_healpix_function_jax_cpu_newenv_vmap.json',
    'jax_healpy_function_gpu_nside1024_halos2000_qdisc200_job6484842.json',
]
rows = []
for name in jax_files:
    payload = load_json(name)
    for row in payload['rows']:
        rows.append({'file': name, **row})
pd.DataFrame(rows)


## pz3 Density Deficit

The exact-cap count reproduces the audit value: 163,967 valid galaxies in the true 600 deg^2 cap. The HOD mean predicts 163,367 galaxies for halo centers in the cap, so the sampled catalog is consistent with the configured HOD. The mismatch is therefore upstream of sampling: either finite-cap density/cosmic variance, the HOD/nbar calibration relative to the Abacus halo catalog, or the target-density convention.

In [ ]:
density = load_json('codex_pz3_density_diagnostic_nside1024_splits_with_hod_mean.json')
density_summary = {
    'valid_buffer': density['density']['n_valid_buffer'],
    'valid_pixel_cap': density['density']['n_valid_in_cap'],
    'valid_exact_cap': density['density']['n_valid_in_exact_cap'],
    'exact_cap_density_deg2': density['density']['surface_density_exact_cap_per_deg2'],
    'retained_target_count': density['density']['target_count_retained_true_z'],
    'ratio_exact_cap_to_retained_target': density['density']['ratio_exact_cap_to_retained_target'],
    'hod_expected_buffer': density['hod_mean']['expected_ngal_buffer'],
    'hod_expected_halo_centers_in_cap': density['hod_mean']['expected_ngal_halo_centers_in_cap'],
    'hod_cap_expected_over_retained_target': density['hod_mean']['ratio_expected_halo_centers_in_cap_to_retained_target'],
}
pd.Series(density_summary)


## Reproduction Commands

```bash
# Function-level healpy and optional jax-healpy diagnostic
/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py \
  benchmark-healpix-functions --config notebooks/xDESI/abacus_paste/stage31_pz3_cap600.selected.yaml \
  --nside 1024 --n-halos 2000 --query-disc-count 200

# Pixel-builder backend comparison
/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py \
  benchmark-pixel-work --config notebooks/xDESI/abacus_paste/stage31_pz3_cap600.selected.yaml \
  --nside 1024 --sample-sizes 50000 --workers 1,8 --pool-chunksizes 128 \
  --single-pixel-angle-factor 0.5 --pixel-backend healpy-buff

# Full pz3 split density plus CPU HOD-mean expectation
/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py \
  diagnose-galaxy-density --config notebooks/xDESI/abacus_paste/stage31_pz3_cap600.selected.yaml \
  --nside 1024 --num-splits 4 --include-hod-mean --hod-platform cpu --hod-chunk-size 500000
```
